In [1]:
import pandas as pd
import numpy as np
import os
import shutil
import re
from pathlib import Path

In [ ]:
def pivot_table(df, index_col, col, val):
    temp_df = df.copy()
    temp_df["Elapsed Minutes"] = pd.to_timedelta(temp_df['Elapsed Time'])
    temp_df["Elapsed Minutes"] = (temp_df['Elapsed Minutes'].dt.total_seconds() / 60).round(2)
    pivot_table_df = pd.pivot_table(temp_df, index=index_col, columns=col, values=val, aggfunc="mean", fill_value=0).stack().reset_index()
    if isinstance(index_col, list):
        group_cols = index_col + [col]
        new_cols = index_col + [col, "Average Elapsed Minutes"]
    else:
        group_cols = [index_col, col]
        new_cols = [index_col, col, "Average Elapsed Minutes"]
    pivot_table_df.columns = new_cols
    pivot_table_df["Average Elapsed Minutes"] = pivot_table_df["Average Elapsed Minutes"].round(2)
    temp_df["Total Failed Jobs"] = (temp_df["Status Code"] > 1).astype(int)
    temp_df["Total Successful Jobs"] = (temp_df["Status Code"] <= 1).astype(int)
    job_status_summary = temp_df.groupby(index_col + [col])[["Failed Jobs", "Successful Jobs"]].sum().reset_index()
    job_latest_success = (temp_df[temp_df["Status Code"] <= 1].groupby(group_cols)["End Time"].max().reset_index().rename(columns={"End Time": "Latest Successful End Time"}))
    pivot_table_df = pivot_table_df.merge(job_status_summary, on=index_col + [col], how="left")
    pivot_table_df = pivot_table_df.merge(job_latest_success, on=group_cols, how="left")
    return pivot_table_df

In [3]:
policies_list_df = pd.read_csv("data\email_dl.csv")

In [83]:
source_folders = [Path(r"C:\Users\User\Desktop\Projects\netbackup_weekly_report\data\May_2026")]

all_dfs = []
all_failed_dfs = []

for source_folder in source_folders:
    report_csv_files = source_folder.rglob("daily_backup_report_*")
    failed_report_csv_files = source_folder.rglob("daily_baselined_failed_backup_report_*")
    for report_csv_file in report_csv_files:
        try:
            df = pd.read_csv(report_csv_file)
            df = df.drop(columns = 'Number of Child Jobs')
            if 'Failure Count' in df.columns and 'Backup Selection' in df.columns:
                df = df.drop(columns=['Failure Count', 'Backup Selection'], errors='ignore')
            all_dfs.append(df)
        except Exception as e:
            print(f"Error : {e}")
    for failed_report_csv_file in failed_report_csv_files:
        try:
            df = pd.read_csv(failed_report_csv_file)
            df = df.drop(columns = 'Number of Child Jobs')
            if 'Backup Selection' in df.columns:
                df = df.drop(columns='Backup Selection', errors='ignore')
            all_failed_dfs.append(df)
        except Exception as e:
            print(f"Error : {e}")

# combine all csv into one dataframe
may_df = pd.concat(all_dfs, ignore_index=True)
may_df["Start Time"] = pd.to_datetime(may_df["Start Time"], errors="coerce")
may_df["End Time"] = pd.to_datetime(may_df["End Time"], errors="coerce")

# combine all csv into one dataframe
may_failed_df = pd.concat(all_failed_dfs, ignore_index=True)
may_failed_df["Start Time"] = pd.to_datetime(may_failed_df["Start Time"], errors="coerce")
may_failed_df["End Time"] = pd.to_datetime(may_failed_df["End Time"], errors="coerce")

Error : No columns to parse from file
Error : No columns to parse from file


C:\Users\User\AppData\Local\Temp\ipykernel_6852\3636673693.py:35: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  may_failed_df["Start Time"] = pd.to_datetime(may_failed_df["Start Time"], errors="coerce")


In [14]:
may_df

,Job,Type,Schedule,Parent Job,Client,Policy,State,Status Code,Start Time,End Time,Elapsed Time
0,8804053,DBBACKUP,-,-,anmirny11.dsta.infra.sg,CATALOG-BACKUP-DTTA,DONE,1,2026-04-30 09:00:00,2026-04-30 09:35:57,35m57s
1,8804054,BACKUP,-,-,anmirny11.dsta.infra.sg,EHAB-DB-SQL-MYClearance-INTEL,DONE,0,2026-04-30 09:00:00,2026-04-30 09:01:47,1m47s
2,8804055,BACKUP,Daily-Log-9am,8804054,ocurewera1.dsta.gov.sg,EHAB-DB-SQL-MYClearance-INTEL,DONE,0,2026-04-30 09:00:06,2026-04-30 09:01:47,1m41s
3,8804056,DBBACKUP,Daily-Catalog,8804053,anmirny11.dsta.infra.sg,CATALOG-BACKUP-DTTA,DONE,0,2026-04-30 09:00:11,2026-04-30 09:06:48,6m37s
4,8804057,BACKUP,Daily-Log-9am,8804055,OCUREWERA1,EHAB-DB-SQL-MYClearance-INTEL,DONE,0,2026-04-30 09:00:22,2026-04-30 09:00:39,17s
...,...,...,...,...,...,...,...,...,...,...,...
103232,8911417,BACKUP,Daily-Log-8am,8911331,OCDMITROV01,EHAB-DB-SQL-WEBAPP1,DONE,0,2026-05-31 08:16:19,2026-05-31 08:16:34,15s
103233,8911418,BACKUP,Daily-Log-8am,8911331,OCDMITROV01,EHAB-DB-SQL-WEBAPP1,DONE,0,2026-05-31 08:16:41,2026-05-31 08:16:56,15s
103234,8911419,BACKUP,Daily-Log-8am,8911331,OCDMITROV01,EHAB-DB-SQL-WEBAPP1,DONE,0,2026-05-31 08:17:04,2026-05-31 08:17:22,18s
103235,8911420,BACKUP,Daily-Log-8am,8911331,OCDMITROV01,EHAB-DB-SQL-WEBAPP1,DONE,0,2026-05-31 08:17:31,2026-05-31 08:17:48,17s


In [82]:
may_failed_df.loc[may_failed_df['Client'] == 'MSG-EXDAG-01.dsta.gov.sg'].sort_values(["Client", "Policy", "Start Time"])

,Job,Type,Schedule,Parent Job,Client,Policy,State,Status Code,Start Time,End Time,Elapsed Time,Failure Count
579,8868641,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All,DONE,10,2026-05-19 17:01:37,2026-05-19 17:26:46,25m9s,1
580,8868642,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All,DONE,10,2026-05-19 17:01:46,2026-05-19 17:07:14,5m28s,1
581,8868687,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All,DONE,10,2026-05-19 17:48:42,2026-05-19 17:56:38,7m56s,1
637,8871834,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All,DONE,150,2026-05-20 08:56:38,2026-05-20 20:31:16,11h34m38s,2
13,8804082,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,2,2026-04-30 10:56:35,2026-04-30 12:02:21,1h5m46s,9
10,8806873,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,2,2026-05-01 03:02:21,2026-05-01 04:13:49,1h11m28s,9
68,8808700,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,2,2026-05-01 19:13:48,2026-05-01 20:25:21,1h11m33s,10
145,8811478,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,130,2026-05-02 11:25:20,2026-05-02 14:09:33,2h44m13s,11
194,8816441,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,130,2026-05-04 01:03:00,2026-05-04 03:32:39,2h29m39s,12
248,8818866,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,150,2026-05-04 18:06:56,2026-05-04 18:45:55,38m59s,13


In [ ]:
parent_jobs = set(may_failed_df["Job"].astype(str))

may_failed_filtered_df = may_failed_df[
    ~may_failed_df["Parent Job"].astype(str).isin(parent_jobs)
].copy()

may_failed_filtered_df["Start Day"] = may_failed_filtered_df["Start Time"].dt.date
may_failed_filtered_df = may_failed_filtered_df.sort_values(["Client", "Policy", "Start Time"])

may_failed_filtered_df = may_failed_filtered_df.drop_duplicates(subset=["Start Day", "Policy", "Client"], keep="first")

may_failed_filtered_df

,Job,Type,Schedule,Parent Job,Client,Policy,State,Status Code,Start Time,End Time,Elapsed Time,Failure Count,Start Day
988,8880734,BACKUP,-,-,AnCova,EHAB-NSOC-SYS-WIN-DLP,DONE,50,2026-05-23 00:06:53,2026-05-23 01:34:50,1h27m57s,1,2026-05-23
986,8880884,BACKUP,-,-,AnGolselDel01,EHAB-NSOC-SYS-WIN-SPLUNK,DONE,58,2026-05-23 01:34:27,2026-05-23 02:32:14,57m47s,1,2026-05-23
158,8813563,BACKUP,-,-,AnGolselDel03,EHAB-NSOC-SYS-WIN-SPLUNK,DONE,69,2026-05-03 01:00:00,2026-05-03 03:09:53,2h9m53s,1,2026-05-03
307,8823993,BACKUP,-,-,AnGolselDel03,EHAB-NSOC-SYS-WIN-SPLUNK,DONE,69,2026-05-06 01:00:00,2026-05-06 03:07:05,2h7m5s,1,2026-05-06
355,8831517,BACKUP,-,-,AnGolselDel03,EHAB-NSOC-SYS-WIN-SPLUNK,DONE,69,2026-05-08 01:00:04,2026-05-08 02:59:55,1h59m51s,1,2026-05-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1218,8887076,BACKUP,-,-,octongariro2.dsta.gov.sg,EHAB-DB-SYS-WIN-ITSMC,DONE,9132,2026-05-25 01:00:01,2026-05-25 01:41:12,41m11s,3,2026-05-25
1284,8906272,BACKUP,-,-,octongariro2.dsta.gov.sg,EHAB-DB-SYS-WIN-ITSMC,DONE,7656,2026-05-30 01:00:00,2026-05-30 01:40:20,40m20s,1,2026-05-30
1300,8910141,BACKUP,-,-,octongariro2.dsta.gov.sg,EHAB-DB-SYS-WIN-ITSMC,DONE,7656,2026-05-31 01:00:00,2026-05-31 01:40:16,40m16s,2,2026-05-31
849,8880755,BACKUP,Daily,-,ocvidin1.dsta.gov.sg,EHAB-DB-SYS-WIN-VMS,DONE,58,2026-05-23 00:06:59,2026-05-23 01:34:38,1h27m39s,1,2026-05-23


In [71]:
may_failed_filtered_df.loc[may_failed_filtered_df['Client'] == 'MSG-EXDAG-01.dsta.gov.sg']

,Job,Type,Schedule,Parent Job,Client,Policy,State,Status Code,Start Time,End Time,Elapsed Time,Failure Count,Start Day
579,8868641,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All,DONE,10,2026-05-19 17:01:37,2026-05-19 17:26:46,25m9s,1,2026-05-19
637,8871834,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All,DONE,150,2026-05-20 08:56:38,2026-05-20 20:31:16,11h34m38s,2,2026-05-20
13,8804082,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,2,2026-04-30 10:56:35,2026-04-30 12:02:21,1h5m46s,9,2026-04-30
10,8806873,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,2,2026-05-01 03:02:21,2026-05-01 04:13:49,1h11m28s,9,2026-05-01
145,8811478,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,130,2026-05-02 11:25:20,2026-05-02 14:09:33,2h44m13s,11,2026-05-02
194,8816441,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,130,2026-05-04 01:03:00,2026-05-04 03:32:39,2h29m39s,12,2026-05-04
315,8821871,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,10,2026-05-05 09:45:55,2026-05-05 11:45:59,2h4s,14,2026-05-05
314,8824792,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,10,2026-05-06 02:45:59,2026-05-06 05:05:55,2h19m56s,14,2026-05-06
358,8829708,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,10,2026-05-07 12:43:50,2026-05-07 14:42:45,1h58m55s,16,2026-05-07
357,8832713,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,2,2026-05-08 05:42:44,2026-05-08 07:24:53,1h42m9s,16,2026-05-08


In [77]:
may_failed_filtered_df_copy = may_failed_filtered_df.copy()
may_failed_filtered_df_copy = may_failed_filtered_df_copy.sort_values(["Client", "Policy", "Start Time"])

may_failed_filtered_df_copy["New Incident"] = (
    may_failed_filtered_df_copy.groupby(["Client", "Policy"])["Failure Count"]
      .diff()
      .ne(1)
)

may_failed_filtered_df_copy["Incident ID"] = (
    may_failed_filtered_df_copy.groupby(["Client", "Policy"])["New Incident"]
      .cumsum()
)

may_failed_filtered_df_copy

,Job,Type,Schedule,Parent Job,Client,Policy,State,Status Code,Start Time,End Time,Elapsed Time,Failure Count,Start Day,New Incident,Incident ID
988,8880734,BACKUP,-,-,AnCova,EHAB-NSOC-SYS-WIN-DLP,DONE,50,2026-05-23 00:06:53,2026-05-23 01:34:50,1h27m57s,1,2026-05-23,True,1
986,8880884,BACKUP,-,-,AnGolselDel01,EHAB-NSOC-SYS-WIN-SPLUNK,DONE,58,2026-05-23 01:34:27,2026-05-23 02:32:14,57m47s,1,2026-05-23,True,1
158,8813563,BACKUP,-,-,AnGolselDel03,EHAB-NSOC-SYS-WIN-SPLUNK,DONE,69,2026-05-03 01:00:00,2026-05-03 03:09:53,2h9m53s,1,2026-05-03,True,1
307,8823993,BACKUP,-,-,AnGolselDel03,EHAB-NSOC-SYS-WIN-SPLUNK,DONE,69,2026-05-06 01:00:00,2026-05-06 03:07:05,2h7m5s,1,2026-05-06,True,2
355,8831517,BACKUP,-,-,AnGolselDel03,EHAB-NSOC-SYS-WIN-SPLUNK,DONE,69,2026-05-08 01:00:04,2026-05-08 02:59:55,1h59m51s,1,2026-05-08,True,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1218,8887076,BACKUP,-,-,octongariro2.dsta.gov.sg,EHAB-DB-SYS-WIN-ITSMC,DONE,9132,2026-05-25 01:00:01,2026-05-25 01:41:12,41m11s,3,2026-05-25,False,1
1284,8906272,BACKUP,-,-,octongariro2.dsta.gov.sg,EHAB-DB-SYS-WIN-ITSMC,DONE,7656,2026-05-30 01:00:00,2026-05-30 01:40:20,40m20s,1,2026-05-30,True,2
1300,8910141,BACKUP,-,-,octongariro2.dsta.gov.sg,EHAB-DB-SYS-WIN-ITSMC,DONE,7656,2026-05-31 01:00:00,2026-05-31 01:40:16,40m16s,2,2026-05-31,False,2
849,8880755,BACKUP,Daily,-,ocvidin1.dsta.gov.sg,EHAB-DB-SYS-WIN-VMS,DONE,58,2026-05-23 00:06:59,2026-05-23 01:34:38,1h27m39s,1,2026-05-23,True,1


In [78]:
may_failed_filtered_df_copy.loc[may_failed_filtered_df_copy['Client'] == 'MSG-EXDAG-01.dsta.gov.sg']

,Job,Type,Schedule,Parent Job,Client,Policy,State,Status Code,Start Time,End Time,Elapsed Time,Failure Count,Start Day,New Incident,Incident ID
579,8868641,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All,DONE,10,2026-05-19 17:01:37,2026-05-19 17:26:46,25m9s,1,2026-05-19,True,1
637,8871834,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All,DONE,150,2026-05-20 08:56:38,2026-05-20 20:31:16,11h34m38s,2,2026-05-20,False,1
13,8804082,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,2,2026-04-30 10:56:35,2026-04-30 12:02:21,1h5m46s,9,2026-04-30,True,1
10,8806873,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,2,2026-05-01 03:02:21,2026-05-01 04:13:49,1h11m28s,9,2026-05-01,True,2
145,8811478,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,130,2026-05-02 11:25:20,2026-05-02 14:09:33,2h44m13s,11,2026-05-02,True,3
194,8816441,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,130,2026-05-04 01:03:00,2026-05-04 03:32:39,2h29m39s,12,2026-05-04,False,3
315,8821871,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,10,2026-05-05 09:45:55,2026-05-05 11:45:59,2h4s,14,2026-05-05,True,4
314,8824792,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,10,2026-05-06 02:45:59,2026-05-06 05:05:55,2h19m56s,14,2026-05-06,True,5
358,8829708,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,10,2026-05-07 12:43:50,2026-05-07 14:42:45,1h58m55s,16,2026-05-07,True,6
357,8832713,BACKUP,-,-,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,DONE,2,2026-05-08 05:42:44,2026-05-08 07:24:53,1h42m9s,16,2026-05-08,True,7


In [79]:
incident_summary = (
    may_failed_filtered_df_copy.groupby(["Client", "Policy", "Incident ID"])
      .agg(
          Start=("Start Time", "min"),
          End=("Start Time", "max"),
          MaxFailureCount=("Failure Count", "max")
      )
      .reset_index()
)

incident_summary

,Client,Policy,Incident ID,Start,End,MaxFailureCount
0,AnCova,EHAB-NSOC-SYS-WIN-DLP,1,2026-05-23 00:06:53,2026-05-23 00:06:53,1
1,AnGolselDel01,EHAB-NSOC-SYS-WIN-SPLUNK,1,2026-05-23 01:34:27,2026-05-23 01:34:27,1
2,AnGolselDel03,EHAB-NSOC-SYS-WIN-SPLUNK,1,2026-05-03 01:00:00,2026-05-03 01:00:00,1
3,AnGolselDel03,EHAB-NSOC-SYS-WIN-SPLUNK,2,2026-05-06 01:00:00,2026-05-06 01:00:00,1
4,AnGolselDel03,EHAB-NSOC-SYS-WIN-SPLUNK,3,2026-05-08 01:00:04,2026-05-08 01:00:04,1
...,...,...,...,...,...,...
365,octongariro1.dsta.gov.sg,EHAB-DB-SYS-WIN-ITSMC,1,2026-05-20 01:00:00,2026-05-23 01:34:27,4
366,octongariro2.dsta.gov.sg,EHAB-DB-SYS-WIN-ITSMC,1,2026-05-23 01:34:27,2026-05-25 01:00:01,3
367,octongariro2.dsta.gov.sg,EHAB-DB-SYS-WIN-ITSMC,2,2026-05-30 01:00:00,2026-05-31 01:00:00,2
368,ocvidin1.dsta.gov.sg,EHAB-DB-SYS-WIN-VMS,1,2026-05-23 00:06:59,2026-05-23 00:06:59,1


In [80]:
incident_summary.loc[incident_summary['Client'] == 'MSG-EXDAG-01.dsta.gov.sg']

,Client,Policy,Incident ID,Start,End,MaxFailureCount
112,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All,1,2026-05-19 17:01:37,2026-05-20 08:56:38,2
113,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,1,2026-04-30 10:56:35,2026-04-30 10:56:35,9
114,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,2,2026-05-01 03:02:21,2026-05-01 03:02:21,9
115,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,3,2026-05-02 11:25:20,2026-05-04 01:03:00,12
116,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,4,2026-05-05 09:45:55,2026-05-05 09:45:55,14
117,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,5,2026-05-06 02:45:59,2026-05-06 02:45:59,14
118,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,6,2026-05-07 12:43:50,2026-05-07 12:43:50,16
119,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,7,2026-05-08 05:42:44,2026-05-08 05:42:44,16
120,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,8,2026-05-09 14:07:40,2026-05-11 01:00:00,19
121,MSG-EXDAG-01.dsta.gov.sg,EHAB-PST-MSG-EXDAG-01-All-Daily,9,2026-05-12 08:30:34,2026-05-12 08:30:34,21


In [74]:
incident_summary.to_csv("failed_streak_may_2.csv", index=False)

In [84]:
policy_df = {}
policy_summary_df = {}

output_path = Path("report_output/")
if not output_path.exists():
    output_path.mkdir()

for pattern in policies_list_df['Policy'].dropna().unique():
    filtered_df_all = may_df[
        may_df['Policy'].astype(str).str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )].copy().sort_values(by=['Client', 'Policy'], ascending=True)
    filtered_df = filtered_df_all.loc[filtered_df_all['Parent Job'] == '-']
    filtered_df = filtered_df.sort_values(by=['Client', 'Policy'], ascending=True)
    filtered_pivot_df = pivot_table(filtered_df.copy(), index_col=["Client", "Policy"], col="State", val="Elapsed Minutes")
    policy_df[pattern] = filtered_df_all
    policy_summary_df[pattern] = filtered_pivot_df

In [ ]:
# for name, df in policy_df.items():
#     df.to_csv(f"{output_path}/{name}_report.csv", index=False)

for name in policy_df.keys():
    destination_file = f"{output_path}/{name}_report.xlsx"
    if os.path.exists(destination_file):
            os.remove(destination_file)
    with pd.ExcelWriter(destination_file, engine="openpyxl") as writer:

        policy_df[name].to_excel(
            writer,
            sheet_name=f"{name}_Report",
            index=False
        )

        policy_summary_df[name].to_excel(
            writer,
            sheet_name=f"{name}_Summary",
            index=False
        )

: 